# 04 — The IF Causal-Work Threshold (v0 toy)

> **Status: v0 TOY.** One rule family, one environment class. Demonstrates the
> intervention protocol end-to-end; does NOT establish IF-H1, which requires
> ≥3 unrelated rule families (rung 274177). See `canon/00-foundations/03-causal-work-principle.md`.

## CONTRACT (v0.1, frozen 2026-07-18 — v0.0's environment was falsified-by-bug: agents
## sensed the world after it moved, making prediction worthless by construction; redesigned
## to act→world-moves→collect BEFORE any sweep was interpreted)

- **PREDICTION**: two separate thresholds exist and DO NOT coincide:
  (1) Π_A = (W_intact − W_scrambled)/C_model crosses 1 at some predictability p*₁;
  (2) memory beats memorylessness (W_intact > W_reactive) only above some p*₂ > p*₁.
  Between them: the **thermodynamic parasite band** — information that is causally
  load-bearing under ablation yet net-negative to possess.
- **BASELINE**: memoryless reactive agent, identical seeded environment.
- **DATA**: seeded synthetic drift-gradient ring world (no external data).
- **PASS**: p*₁ and p*₂ both exist with p*₂ > p*₁ and a non-empty parasite band.
- **FALSIFIER**: Π_A > 1 everywhere/nowhere, or the two thresholds coincide
  (ablation value ≡ competitive value → the Π_A construct adds nothing beyond
  a simple pays/doesn't-pay comparison in this family).

In [ ]:
import numpy as np

SEED = 65537
C_SENSE, C_MEMORY, C_MOVE = 0.010, 0.020, 0.005

# Ring world: a narrow resource hill whose peak drifts 1 cell/step; drift direction
# persists with probability p. The agent commits its move BEFORE the world steps
# (sense→act→world-moves→collect), so tracking the CURRENT peak always lags by one
# cell — only a correct drift PREDICTION can ride the peak. Memory is the only
# place a drift estimate can live; scrambling replaces it with marginal-preserving noise.

def run(predictability, use_memory, scramble=False, steps=4000, seed=SEED):
    rng = np.random.default_rng(seed)          # world stream
    srng = np.random.default_rng(seed + 1)     # scrambler stream (independent)
    n = 64
    peak, direction = 0, 1
    pos, gathered, model_cost = 0, 0.0, 0.0
    believed, prev_peak = 1, 0

    def resource(p_, peak_):
        d = min(abs(p_ - peak_), n - abs(p_ - peak_))
        return max(0.0, 1.0 - d / 8.0)

    for _ in range(steps):
        gathered -= C_SENSE                      # peak-position sensor
        if use_memory:
            model_cost += C_MEMORY
            gathered -= C_MEMORY                 # ledger debit for keeping the model
            delta = (peak - prev_peak + n//2) % n - n//2
            if delta: believed = 1 if delta > 0 else -1
            if scramble: believed = srng.choice([-1, 1])
            target = (peak + believed) % n       # intercept the PREDICTED next peak
        else:
            target = peak                        # chase the CURRENT peak
        prev_peak = peak
        offset = (target - pos + n//2) % n - n//2
        if offset:
            gathered -= C_MOVE
            pos = (pos + (1 if offset > 0 else -1)) % n
        if rng.random() > predictability:        # world moves AFTER the commit
            direction = -direction
        peak = (peak + direction) % n
        gathered += resource(pos, peak)
    return gathered, model_cost


In [ ]:
rows = []
for p in np.linspace(0.5, 0.995, 12):
    w_intact, c_model = run(p, use_memory=True)
    w_scram, _ = run(p, use_memory=True, scramble=True)
    w_reactive, _ = run(p, use_memory=False)
    pi_a = (w_intact - w_scram) / c_model
    rows.append((p, w_intact, w_scram, w_reactive, pi_a))
    print(f"p={p:0.3f}  W_intact={w_intact:8.1f}  W_scrambled={w_scram:8.1f}  "
          f"W_reactive={w_reactive:8.1f}  Pi_A={pi_a:6.3f}  mem_adv={w_intact-w_reactive:+8.1f}")

rows = np.array(rows)
cross1 = rows[np.argmax(rows[:,4] > 1.0), 0] if (rows[:,4] > 1.0).any() else None
cross2 = rows[np.argmax(rows[:,1] > rows[:,3]), 0] if (rows[:,1] > rows[:,3]).any() else None
parasite = rows[(rows[:,4] > 1.0) & (rows[:,1] < rows[:,3])]
print()
print(f"p*1 (Pi_A crosses 1):            {cross1}")
print(f"p*2 (memory beats memoryless):   {cross2}")
print(f"thermodynamic parasite band rows (Pi_A>1 AND worse than memoryless): {len(parasite)}")
verdict = (cross1 is not None and cross2 is not None and cross2 > cross1 and len(parasite) > 0)
print()
print("PASS — two distinct thresholds + parasite band" if verdict
      else "FALSIFIER FIRED — log in SCOREBOARD kill log")

## Result (v0.1 run, seed 65537)

Observed: p*₁ ≈ 0.64 (Π_A crosses 1) but p*₂ ≈ 0.995 (memory finally beats
memorylessness). Between them, a wide **thermodynamic parasite band**: scrambling
the drift belief costs hundreds of work units (the information is causally
load-bearing), yet the agent would be better off with no memory at all —
an actively-wrong belief misroutes the agent, so the ablation delta OVERSTATES
the value of possessing memory.

Why this matters beyond the toy:
- It is a concrete existence proof of the Gemini-panel 'net-negative thermodynamic
  parasite' — information high in KW-style semantic value (it predicts the drift!)
  that a full-cost IF audit rejects.
- It sharpens IF-H1: the agency threshold must be stated against the RIGHT baseline.
  Π_A > 1 measures causal load-bearing; agency-that-pays requires the competitive
  criterion. The canonical formulation needs both inequalities.

## Interpretation guardrails

- Designed memory mechanism → demonstrates the PROTOCOL, not IF-H1 universality
  (that needs emergent structures across ≥3 unrelated rule families, Conway gate).
- The scramble preserves the marginal distribution of the belief variable; intact and
  scrambled runs pay identical memory costs — only correlations differ. That is the point.
- Next: `04a_kw_vs_if_divergence`, `04c_fep_cost_audit`, `04d_maxwell_demon_landauer`.